In [1]:
import pandas as pd
import os
import sys

# --- Configuration ---
# Update this path to your MIMIC-Extract HDF5 file.
HDF_FILE_PATH = '../data/raw/all_hourly_data.h5'
# Directory to save feature CSV files
OUTPUT_DIR = '../data/processed/feature_names'

def save_feature_names(df, table_key):
    """
    Save feature names from a DataFrame to a CSV file.
    
    Args:
        df (pd.DataFrame): DataFrame containing the data
        table_key (str): The HDF5 table key/name
    """
    # Clean the table key to make a valid filename
    table_name = table_key.replace('/', '_').strip('_')
    if not table_name:
        table_name = 'unnamed_table'
    
    # Extract column names
    if hasattr(df.columns, 'levels'):
        # Multi-level columns - flatten them
        if len(df.columns.levels) > 1:
            feature_names = []
            for col in df.columns:
                if isinstance(col, tuple):
                    # Join tuple elements with underscore, skip NaN values
                    col_name = '_'.join([str(c) for c in col if pd.notna(c) and str(c) != 'nan'])
                else:
                    col_name = str(col)
                feature_names.append(col_name)
        else:
            feature_names = [str(col) for col in df.columns.get_level_values(0)]
    else:
        # Single-level columns
        feature_names = [str(col) for col in df.columns]
    
    # Create DataFrame with feature names
    features_df = pd.DataFrame({'feature_name': feature_names})
    
    # Save to CSV
    output_file = os.path.join(OUTPUT_DIR, f'{table_name}_features.csv')
    features_df.to_csv(output_file, index=False)
    print(f"  --> Saved {len(feature_names)} feature names to: {output_file}")

def inspect_hdf5_file_robustly(filepath, save_features=True):
    """
    Reads and displays the head of each table in a MIMIC-Extract HDF5 file,
    handling potential format incompatibilities. Optionally saves feature names to CSV files.

    Args:
        filepath (str): The full path to the .h5 file.
        save_features (bool): Whether to save feature names to CSV files.
    """
    if not os.path.exists(filepath):
        print(f"--- CRITICAL ERROR: File not found ---")
        print(f"The path '{filepath}' does not exist. Please check your configuration.")
        sys.exit(1) # Exit with an error code

    # Create output directory for feature names if saving is enabled
    if save_features:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"Feature names will be saved to: {OUTPUT_DIR}\n")

    print(f"--- Begin HDF5 File Inspection ---")
    print(f"Target file: {filepath}\n")
    print(f"Pandas version being used: {pd.__version__}\n")

    try:
        # Use HDFStore in read-only mode ('r'). This is safer.
        with pd.HDFStore(filepath, 'r') as store:
            # Discover the keys (table names) within the file.
            all_keys = store.keys()
            if not all_keys:
                print("--- WARNING: The HDF5 file is empty or unreadable. ---")
                return

            print(f"Discovered the following tables (keys) in the file: {all_keys}\n")

            for key in all_keys:
                print(f"==========================================================")
                print(f"Table: '{key}'")
                print(f"==========================================================")

                try:
                    # METHOD 1 (Ideal): Memory-efficient read of the first 5 rows.
                    # This uses the HDF5 file's query engine (where clause) and is
                    # the correct way to handle large tables.
                    df_head = store.select(key, start=0, stop=5)
                    print(df_head)
                    
                    # Save feature names if requested
                    if save_features:
                        save_feature_names(df_head, key)

                except (TypeError, IndexError, ValueError) as e:
                    # RATIONALE: These errors are symptomatic of a format mismatch.
                    # The query engine in modern pandas cannot interpret the file's
                    # index or table format correctly to perform a ranged select.
                    print(f"--> INFO: Standard query failed for '{key}'. This is expected for older HDF5 formats.")
                    print(f"--> Error was: {type(e).__name__}: {e}")
                    print("--> Attempting alternative full-select method (still efficient if it works)...")

                    try:
                        # METHOD 2 (Fallback): Select the entire table (efficiently) and then
                        # take the head in-memory. This often works when start/stop fails.
                        df_full = store.select(key)
                        df_head = df_full.head(5)
                        print(df_head)
                        
                        # Save feature names if requested
                        if save_features:
                            save_feature_names(df_full, key)
                    except Exception as fallback_e:
                        print(f"--- FAILURE ---")
                        print(f"All efficient read methods have failed for table '{key}'.")
                        print(f"This indicates a severe incompatibility or file corruption.")
                        print(f"Fallback Error: {fallback_e}")

                except Exception as e:
                    print(f"--- UNEXPECTED FAILURE ---")
                    print(f"A non-standard error occurred while reading '{key}': {e}")

                print("\n")

    except Exception as e:
        print(f"--- A CRITICAL Error Occurred Opening the HDF5 Store ---")
        print(f"Could not process the HDF5 file itself. It may be corrupted or not a valid HDF5 file.")
        print(f"Error: {e}")


In [2]:

inspect_hdf5_file_robustly(HDF_FILE_PATH, save_features=True)

Feature names will be saved to: ../data/processed/feature_names

--- Begin HDF5 File Inspection ---
Target file: ../data/raw/all_hourly_data.h5

Pandas version being used: 1.3.5

Discovered the following tables (keys) in the file: ['/codes', '/interventions', '/patients', '/vitals_labs', '/vitals_labs_mean', '/patients/meta/values_block_6/meta', '/patients/meta/values_block_5/meta', '/patients/meta/values_block_4/meta', '/patients/meta/values_block_0/meta']

Table: '/codes'
--> INFO: Standard query failed for '/codes'. This is expected for older HDF5 formats.
--> Error was: ValueError: On level 1, code value (-72) < -1
--> Attempting alternative full-select method (still efficient if it works)...
                                                                               icd9_codes
subject_id hadm_id icustay_id hours_in                                                   
3          145834  211552     0         [0389, 78559, 5849, 4275, 41071, 4280, 6826, 4...
                        

In [10]:
import pandas as pd

# Path to the Parquet file
parquet_path = "/Users/aaronge/Documents/EHR Embeddings/data/meds_cohort_split_filtered/data/test/data.parquet"


# Load the Parquet file
df = pd.read_parquet(parquet_path)

# 1. How many patients are in this file?
num_patients = df['patient_id'].nunique()
print(f"Number of unique patients in the file: {num_patients}")

# 2. What unique events are encoded for these patients?
# Let's inspect the columns and unique event types
print("\nColumns in the dataframe:")
print(df.columns.tolist())

# If there is a 'code' column, show unique event codes
if 'code' in df.columns:
    unique_codes = df['code'].unique()
    print(f"\nNumber of unique event codes: {len(unique_codes)}")
    print("Sample event codes:", unique_codes[:10])
else:
    print("\nNo 'code' column found. Cannot display unique event codes.")

# If there is a 'time' column, show how many unique event times there are
if 'time' in df.columns:
    unique_times = df['time'].nunique()
    print(f"\nNumber of unique event times: {unique_times}")
    print("Sample event times:", df['time'].drop_duplicates().head(5).tolist())

# Optionally, show a sample of events for the first patient
first_patient_id = df['patient_id'].iloc[0]
print(f"\nSample events for patient_id {first_patient_id}:")
print(df[df['patient_id'] == first_patient_id].head())



Number of unique patients in the file: 5986

Columns in the dataframe:
['patient_id', 'event_id', 'time', 'code', 'numeric_value', 'text_value', 'datetime_value']

Number of unique event codes: 3821
Sample event codes: <StringArray>
[     'ICD9CM/99604',       'ICD9CM/4271',       'ICD9CM/4280',
      'ICD9CM/42731',      'ICD9CM/41401',        'ICD9CM/412',
       'ICD9CM/5939',       'ICD9CM/2720',      'ICD9CM/60000',
 'vitals_labs/index']
Length: 10, dtype: string

Number of unique event times: 486800
Sample event times: [Timestamp('2126-05-06 15:16:00'), Timestamp('2126-05-06 16:16:00'), Timestamp('2126-05-06 17:16:00'), Timestamp('2126-05-06 18:16:00'), Timestamp('2126-05-06 19:16:00')]

Sample events for patient_id 26:
   patient_id  event_id                time          code  numeric_value  \
0          26         0 2126-05-06 15:16:00  ICD9CM/99604            NaN   
1          26         1 2126-05-06 15:16:00   ICD9CM/4271            NaN   
2          26         2 2126-05-06 1

In [ ]:
# 'vitals_labs' refers to a type of event code, not a column.
# Let's explore the unique event codes for a particular patient and filter those related to 'vitals_labs'.

# Show all unique codes for the first patient
patient_df = df[df['patient_id'] == first_patient_id]
unique_codes_patient = patient_df['code'].unique()
print(f"\nNumber of unique event codes for patient_id {first_patient_id}: {len(unique_codes_patient)}")
print("Sample event codes for this patient:", unique_codes_patient[:10])

# Filter codes that contain 'vitals_labs' (case-insensitive)
vitals_labs_codes = [code for code in unique_codes_patient if 'vitals_labs' in str(code).lower()]
print(f"\nUnique 'vitals_labs' codes for patient_id {first_patient_id}:")
print(vitals_labs_codes)
print(f"Number of unique 'vitals_labs' codes: {len(vitals_labs_codes)}")

# Save the patient's full data as a JSON file, handling non-serializable types (e.g., Timestamp)
import json
import numpy as np

def json_serializable(obj):
    """Convert non-serializable objects (like Timestamp, np types) to serializable formats."""
    import pandas as pd
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (pd.Timestamp, pd.Timedelta)):
        return str(obj)
    elif isinstance(obj, (pd.NaT.__class__, type(None))):
        return None
    return str(obj)

patient_json_path = f"patient_{first_patient_id}_full_data.json"
# Convert DataFrame to records (list of dicts) for JSON serialization
patient_records = patient_df.to_dict(orient='records')
with open(patient_json_path, 'w', encoding='utf-8') as f:
    json.dump(patient_records, f, ensure_ascii=False, indent=2, default=json_serializable)

print(f"\nSaved full data for patient_id {first_patient_id} to '{patient_json_path}'")



Number of unique event codes for patient_id 26: 44
Sample event codes for this patient: <StringArray>
[     'ICD9CM/99604',       'ICD9CM/4271',       'ICD9CM/4280',
      'ICD9CM/42731',      'ICD9CM/41401',        'ICD9CM/412',
       'ICD9CM/5939',       'ICD9CM/2720',      'ICD9CM/60000',
 'vitals_labs/index']
Length: 10, dtype: string

Unique 'vitals_labs' codes for patient_id 26:
['vitals_labs/index', 'vitals_labs/anion gap', 'vitals_labs/bicarbonate', 'vitals_labs/blood urea nitrogen', 'vitals_labs/chloride', 'vitals_labs/creatinine', 'vitals_labs/diastolic blood pressure', 'vitals_labs/glucose', 'vitals_labs/heart rate', 'vitals_labs/hematocrit', 'vitals_labs/mean blood pressure', 'vitals_labs/partial thromboplastin time', 'vitals_labs/potassium', 'vitals_labs/prothrombin time inr', 'vitals_labs/prothrombin time pt', 'vitals_labs/respiratory rate', 'vitals_labs/sodium', 'vitals_labs/systolic blood pressure', 'vitals_labs/glascow coma scale total', 'vitals_labs/oxygen saturatio

In [10]:
import pickle

# Path to the file
pkl_path = "/Users/aaronge/Documents/EHR Embeddings/notebooks/Phase 1 and 2/phase_1_outputs/preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_X_val.pkl"

# Load the data and print the header (column names) if possible
with open(pkl_path, "rb") as f:
    data = pickle.load(f)
    print("Type of loaded object:", type(data))
    # If it's a DataFrame, print columns
    try:
        import pandas as pd
        if isinstance(data, pd.DataFrame):
            print("Header (columns):")
            print(data.columns.tolist())
        elif isinstance(data, (list, tuple)) and len(data) > 0:
            first_row = data[0]
            if isinstance(first_row, dict):
                print("Header (keys of first row):")
                print(list(first_row.keys()))
            elif hasattr(first_row, '_fields'):  # namedtuple
                print("Header (namedtuple fields):")
                print(first_row._fields)
            else:
                print("First row type:", type(first_row))
                print("First row:", first_row)
        elif isinstance(data, dict):
            print("Header (top-level keys):")
            print(list(data.keys()))
        else:
            print("Cannot determine header structure for this object.")
    except Exception as e:
        print(f"Error while trying to print header: {e}")

Type of loaded object: <class 'pandas.core.frame.DataFrame'>
Header (columns):
['age', 'alanine aminotransferase_mean_count', 'alanine aminotransferase_mean_last', 'alanine aminotransferase_mean_mean', 'alanine aminotransferase_mean_slope_24h', 'albumin ascites_mean_count', 'albumin ascites_mean_last', 'albumin pleural_mean_count', 'albumin pleural_mean_last', 'albumin urine_mean_count', 'albumin urine_mean_last', 'albumin_mean_count', 'albumin_mean_last', 'alkaline phosphate_mean_count', 'alkaline phosphate_mean_last', 'alkaline phosphate_mean_mean', 'alkaline phosphate_mean_slope_24h', 'anion gap_mean_count', 'anion gap_mean_last', 'anion gap_mean_mean', 'anion gap_mean_slope_24h', 'anion gap_mean_stddev_24h', 'asparate aminotransferase_mean_count', 'asparate aminotransferase_mean_last', 'asparate aminotransferase_mean_mean', 'asparate aminotransferase_mean_slope_24h', 'basophils_mean_count', 'basophils_mean_last', 'basophils_mean_mean', 'basophils_mean_slope_24h', 'bicarbonate_mean_